In [ ]:
dbutils.widgets.removeAll()

from pyspark.sql import functions as F

In [ ]:
dbutils.widgets.text("catalogo", "catalog_smartdata_final")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")
dbutils.widgets.text("storageName", "adlssmartdata2023")
dbutils.widgets.dropdown("writeMode", "overwrite", ["overwrite", "append"])

In [ ]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")
storage_name = dbutils.widgets.get("storageName")
write_mode = dbutils.widgets.get("writeMode")

tbl_population_clean_silver = f"{catalogo}.{esquema_source}.ev_population_clean"
tbl_vehicle_enriched_silver = f"{catalogo}.{esquema_source}.ev_vehicle_enriched"

tbl_golden_popularity = f"{catalogo}.{esquema_sink}.golden_ev_model_popularity"
tbl_golden_price_range = f"{catalogo}.{esquema_sink}.golden_make_price_range"
tbl_golden_state = f"{catalogo}.{esquema_sink}.golden_state_distribution"
tbl_golden_range_regs = f"{catalogo}.{esquema_sink}.golden_range_vs_registrations"
tbl_golden_ratio = f"{catalogo}.{esquema_sink}.golden_make_range_price_ratio"

In [ ]:
df_population_silver = spark.table(tbl_population_clean_silver)
df_vehicle_enriched = spark.table(tbl_vehicle_enriched_silver)

In [ ]:
df_golden_popularity = (
    df_population_silver.groupBy("make", "model")
    .agg(F.count(F.lit(1)).alias("vehicles_registered"))
    .withColumn("updated_ts", F.current_timestamp())
)

df_golden_price_range = (
    df_population_silver.groupBy("make")
    .agg(
        F.avg("base_msrp").alias("avg_price"),
        F.avg("electric_range_km").alias("avg_range"),   # km (corregido de millas)
        F.count(F.lit(1)).alias("total_vehicles")
    )
    .withColumn("updated_ts", F.current_timestamp())
)

df_golden_state = (
    df_population_silver.groupBy("state")
    .agg(F.count(F.lit(1)).alias("total_ev"))
    .withColumn("updated_ts", F.current_timestamp())
)

df_golden_range_regs = (
    df_vehicle_enriched.groupBy("make", "model", "specs_range_km")
    .agg(F.count(F.lit(1)).alias("vehicles_registered"))
    .withColumn("updated_ts", F.current_timestamp())
)

df_golden_ratio = (
    df_vehicle_enriched.filter(F.col("base_msrp") > 0)
    .groupBy("make")
    .agg(
        F.avg("specs_range_km").alias("avg_specs_range_km"),
        F.avg("base_msrp").alias("avg_price")
    )
    .withColumn("range_per_price_ratio", F.col("avg_specs_range_km") / F.col("avg_price"))
    .withColumn("updated_ts", F.current_timestamp())
)


In [ ]:
df_golden_popularity.write.mode(write_mode).option("overwriteSchema", "true").saveAsTable(tbl_golden_popularity)
df_golden_price_range.write.mode(write_mode).option("overwriteSchema", "true").saveAsTable(tbl_golden_price_range)
df_golden_state.write.mode(write_mode).option("overwriteSchema", "true").saveAsTable(tbl_golden_state)
df_golden_range_regs.write.mode(write_mode).option("overwriteSchema", "true").saveAsTable(tbl_golden_range_regs)
df_golden_ratio.write.mode(write_mode).option("overwriteSchema", "true").saveAsTable(tbl_golden_ratio)

print(f"Golden popularity: {df_golden_popularity.count()}")
print(f"Golden price/range: {df_golden_price_range.count()}")
print(f"Golden state distribution: {df_golden_state.count()}")
print(f"Golden range vs registrations: {df_golden_range_regs.count()}")
print(f"Golden make ratio: {df_golden_ratio.count()}")

In [ ]:
display(spark.table(tbl_golden_popularity).orderBy(F.desc("vehicles_registered")).limit(20))
display(spark.table(tbl_golden_price_range).orderBy(F.desc("avg_range")).limit(20))